# 03_02_Building Dimension Train Service

In [1]:
from pathlib import Path

import pandas as pd

In [2]:
PATH_INTERMEDIATE = Path("../data/intermediate/")
PATH_PROCESSED = Path("../data/processed/")

# 2. DIMENSION TRAIN SERVICE

**DESCRIPTION**

`Dim_Train_Service` is a **derived dimension** built from the Infrabel `punctuality` dataset, extracting the **unique combinations** of the following columns: `TRAIN_SERV`, `RELATION`, `RELATION_DIRECTION`, and `DIRECTION_IS_INFERRED`.

It has no natural Business Key. Its **Primary Key** is a **Surrogate Key generated from the unique combination** of the aforementioned four `punctuality` columns. 

Once the dimension is loaded in SQL Server, this Surrogate Key will be the **Foreign Key** between `Dim_Train_Service` and `Fact_Punctuality`.

Its grain is **one row for each train service with a specific terminus, if any terminus is provided**.

It contains **526 entries**. 

In the star schema data warehouse, `Dim_Train_Service` is instrumental to test **the project's second hypothesis** (see **README**), as it enables **punctuality analyses by train route**. 

<br>

**ATTRIBUTES**

`Dim_Train_Service` has 8 attributes:

* `SK_Train_Service` (Int64): Dimension Surrogate Key.

* `Operator_Train_Service` (string): The train service operator - national or international. Contains four classes: `"SNCB/NMBS"`, `"THI-FACT"`, `"EURSLEEPER"`, and `"EUROSTARFR"`.

* `Relation_Category` (string): Train service categories containing 11 classes (see **NOTE** below). 

* `Relation` (string): The code of any train service without direction or terminus.

* `Relation_Direction` (string): The code and name of any train service, with direction and terminus, if provided.

* `Direction_Is_Inferred` (Int64): A binary column flagging inferred `Relation_direction` values, as established in the *02_05_handling_missing_values_relation_directon_punctuality* notebook.

* `Is_Local` (Int64): A binary column that flags local train services, i.e. `"S"` and `"L"` relation categories.

* `Is_National` (Int64): A binary column that flags national train services, i.e. `"IC"`, `"S"`, `"L"`, `"P"`, and `"EXTRA"` rows that correspond to a `"SNCB/NMBS"` value in the `Operator_train_service` column. By extension, this column enables to filter out international train (i.e. `"ICE"`, `"INT"`, `"THAL"`, `"EURST"`, and `"TGV"`).

<br>

**SOURCES**

* Being derived from `punctuality`, the only sources for `Dim_Train_Service` are the Infrabel monthly `punctuality` raw datasets, concatenated in `punctuality_raw` for the 2024-2025 period.  

<br>

**BUILDING PROCESS**

1) `punctuality_cleaned.parquet` is imported as `punctuality`. From this DataFrame, each unique combination of values of the `TRAIN_SERV`, `RELATION`, `RELATION_DIRECTION`, and `DIRECTION_IS_INFERRED` columns is extracted to build the `train_services` DataFrame, to which a **Surrogate Key** is assigned: `SK_Train_Service` (**Section 2.1.**).

2) From `RELATION`, **11 train service categories** are extracted in the `Relation_Category` new column (**Section 2.2.1.**).

3) A `Is_Local` column is created. It flags the `"S"` and `"L"` categories when `"SNCB/NMBS"` is the operator (**Section 2.2.2.**).

4) A `Is_National` column is created. It flags the `"IC"`, `"S"`, `"L"`, `"P"`, and `"EXTRA"` categories when `"SNCB/NMBS"` is the operator (**Section 2.2.3.**).

5) `train_services` is refined and renamed as `Dim_Train_Service`, after `int64` data types are cast to `Int64` and columns are renamed and reorganised (**Section 2.3.**).

6) `Dim_Train_Service` is exported to `../data/processed/` directory. The export result is verified (**Section 2.4.**).

<br>

**NOTE - Relation categories:**

* As explained in Section 4.1.2. of *02_04_profiling_and_cleaning_punctuality*, the 171 `RELATION` unique values can be regrouped into 11 categories:

| Relation category | SNCB Documentation                                                          |
|-------------------|-----------------------------------------------------------------------------|
|	IC              | InterCity trains in Belgium                                                 |
|   S               | Suburban local trains in the 5 main Belgian railway hubs: Brussels, Antwerp, Liège, Ghent, and Charleroi  |
|	L               | Local trains, some renamed "S" since 2015                                   |
|	P               | Additional trains during peak hours or from university cities on weekend    |
|	EXTRA           | Additional trains in case of incident or significant delays on the network  |
|	ICE             | InterCityExpress from German cities, operated by Deutsche Bahn              |
|	INT             | Other international trains                                                  |
|	THAL            | Thalys trains between Brussels and Paris                                    |
|	EURST           | Eurostar trains between London and Brussels, Paris or Amsterdam             |
|	TGV             | French High-Speed Train (Train à Grande Vitesse)                            |
|	CHARTER         | Trains on demand for specific events (e.g. Tomorrowland festival, Venice Simplon Orient Express)                                                                                             |

* These categories are fundamental **to test the project's second hypothesis - the route disparity bias**: if punctuality perception strongly varies depending on the passenger's regular route - some heavier delayed than other -, `Dim_Train_Service` must identify deviations between route categories, not only between specific train services. 

* For instance, suburban local trains have been gradually relabelled from `"L"` to `"S"` train services over the last 11 years. Within the `punctuality` dataset, several train services still carried their older labels and were relabelled in the 2024-2025 periods (e.g. `"L C3-2"` as `"S63"`, `"L C2-2"` as `"S62-2"`, and `"L B5-1"` as `"S5-1"`). Therefore, filtering these train services by general category instead of by `"RELATION"` values will enables analysing `"S"` and `"L"` services together, grouped within the `Is_Local` column.  

* Furthermore, **International train services** have structurally higher delays than national services due to longer travel durations (e.g. **FRANCFORT -> BRUXELLES-CENTRAL** takes 3h24 on average). Although there are not aberrant values, they may be treated as **outliers**. Therefore, grouping them within the `Is_National` column will enable filtering them out.  
Unless otherwise stated, the international services will be filtered out in future Power BI dashboards, as this project focuses on Belgian railway punctuality.

* `"P"` and `"EXTRA"` catogories are retained for further analysis. Although they are not regular train services such as `"IC"` or `"S"`, they are added on the network precisely when delays are higher: peak hours, incidents, significant delays, etc. Therefore, they may be significant for punctuality analyses.

* International `"EXTRA"` trains, if any, are not explicitly excluded from `Is_National` because identifying them would require finding their starting and ending stations by grouping 45M rows by train number, date, and time, then reconstructing their route and inferring whether their endpoints are border stations. This type of in-depth analysis requires the data warehouse to be already built.

* As a train-on-demand service for specific events, `"CHARTER"` is out of the scope of this analysis and will be filtered out of future dashboards and reports.     


## 2.1. Extracting `punctuality` Unique Combinations and Assigning Surrogate Key

* Each unique combination of values of `TRAIN_SERV`, `RELATION`, `RELATION_DIRECTION`, and `DIRECTION_IS_INFERRED` is extracted from `punctuality`, creating the `train_services` DataFrame.

* A new `SK_Train_service` column is added, containing the `train_services` dimension Surrogate Key.

In [3]:
punctuality = pd.read_parquet(PATH_INTERMEDIATE / "punctuality_cleaned.parquet")

In [4]:
punctuality.columns

Index(['DATDEP', 'TRAIN_NO', 'RELATION', 'TRAIN_SERV', 'PTCAR_NO',
       'LINE_NO_DEP', 'REAL_TIME_ARR', 'REAL_TIME_DEP', 'PLANNED_TIME_ARR',
       'PLANNED_TIME_DEP', 'DELAY_ARR', 'DELAY_DEP', 'RELATION_DIRECTION',
       'PTCAR_LG_NM_NL', 'LINE_NO_ARR', 'PLANNED_DATE_ARR', 'PLANNED_DATE_DEP',
       'REAL_DATE_ARR', 'REAL_DATE_DEP', 'DIRECTION_IS_INFERRED',
       'RELATION_DIRECTION_MISSING_DATE'],
      dtype='str')

In [5]:
train_services = (
    punctuality[["TRAIN_SERV", "RELATION", "RELATION_DIRECTION","DIRECTION_IS_INFERRED"]]
).drop_duplicates().sort_values("RELATION_DIRECTION").reset_index(drop=True)

In [6]:
train_services.head()

,TRAIN_SERV,RELATION,RELATION_DIRECTION,DIRECTION_IS_INFERRED
0,SNCB/NMBS,CHARTER,CHARTER,1
1,THI-FACT,EURST,EURST: AMSTERDAM CENTRAAL -> LONDON-ST-PANCRAS...,0
2,THI-FACT,EURST,EURST: AMSTERDAM CENTRAAL -> MARNE-LA-VALLEE-C...,0
3,THI-FACT,EURST,EURST: AMSTERDAM CENTRAAL -> PARIS-NORD,0
4,THI-FACT,EURST,EURST: KOLN HBF -> PARIS-NORD,0


In [7]:
train_services.info()

<class 'pandas.DataFrame'>
RangeIndex: 526 entries, 0 to 525
Data columns (total 4 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   TRAIN_SERV             526 non-null    string
 1   RELATION               526 non-null    string
 2   RELATION_DIRECTION     526 non-null    string
 3   DIRECTION_IS_INFERRED  526 non-null    Int64 
dtypes: Int64(1), string(3)
memory usage: 40.9 KB


In [8]:
train_services.isnull().sum()

TRAIN_SERV               0
RELATION                 0
RELATION_DIRECTION       0
DIRECTION_IS_INFERRED    0
dtype: int64

In [9]:
train_services.nunique()

TRAIN_SERV                 4
RELATION                 171
RELATION_DIRECTION       515
DIRECTION_IS_INFERRED      2
dtype: int64

In [10]:
train_services["SK_Train_Service"] = train_services.index + 1

## 2.2. Adding Derived Columns

### 2.2.1. Extracting `Relation_Category` from `RELATION`

* **11 relation categories** are extracted from `RELATION`. The definitions of these 11 categories are explained in the introduction of this notebook (see **NOTE - Relation categories**).

* Comparing those with the four train operators (i.e. the `TRAIN_SERV` classes) reveals an inconsistency: a single local train service `"L"`, with a unique `TRAIN_NO` value (`1279`), was assigned to the international operator `"EURSLEEPER"` on 9 March 2025, starting from the **VISÉ** border station, and ending at **BRUSSELS-CENTRAL**, which is not consistant with a local route.

* This comparation also reveals that the `"SNCB/NMBS"` national operator manages both national and international train services, while the international operators only manage international train services (except for the inconsistency previously noted, which is excluded from the following table):

| Train operator | National categories  | International categories          | Total rows |
|----------------|----------------------|-----------------------------------|------------|
| SNCB/NMBS      | IC, S, L, P, EXTRA   | INT, THAL, EURST, TGV, ICE, EXTRA | 502        |
| THI-FACT       | None                 | THAL, EURST, TGV, EXTRA           | 17         |
| EURSLEEPER     | None                 | INT                               | 5          |
| EUROSTARFR     | None                 | THAL, EXTRA                       | 2          |

* `EXTRA` trains can be both national or international. Nevertheless, as established in **NOTE - Relation categories**, this category is included within `Is_National` when the operator is `SNCB/NMBS`, because identifying the international `SNCB/NMBS` `EXTRA` trains would require in-depth analysis feasible only once the data warehouse is built.

* `"CHARTER"` trains are on-demand services requested by private entities (companies, organisations, youth groups, etc.) for specific events and provided by `SNCB/NMBS`. As established in **NOTE - Relation categories**, this category is not relevant to the project analyses and is therefore excluded from `Is_National`.

In [11]:
train_services["Relation_Category"] = train_services["RELATION"].str.extract(r'^([A-Z]+)', expand=False)

In [12]:
train_services[["TRAIN_SERV", "Relation_Category"]].value_counts().sort_index(ascending=False)

TRAIN_SERV  Relation_Category
THI-FACT    THAL                   7
            TGV                    1
            EXTRA                  1
            EURST                  8
SNCB/NMBS   THAL                   2
            TGV                    2
            S                    105
            P                      1
            L                    251
            INT                   11
            ICE                    2
            IC                   125
            EXTRA                  1
            EURST                  1
            CHARTER                1
EURSLEEPER  L                      1
            INT                    4
EUROSTARFR  THAL                   1
            EXTRA                  1
Name: count, dtype: int64

In [13]:
eursleep_with_l_cat = (
    punctuality.loc[(punctuality["TRAIN_SERV"] == "EURSLEEPER") & 
                   (punctuality["RELATION"] == "L")]
)
eursleep_with_l_cat[["DATDEP", "TRAIN_NO", "TRAIN_SERV", "RELATION"]].drop_duplicates().reset_index(drop=True)

,DATDEP,TRAIN_NO,TRAIN_SERV,RELATION
0,2025-03-09,1279,EURSLEEPER,L


In [14]:
eursleep_l_cat = eursleep_with_l_cat["PTCAR_LG_NM_NL"].reset_index(drop=True).to_list()
eursleep_l_cat[0]

'VISE'

As shown in the outputs above: 
* The single local train service `"L"`, with `TRAIN_NO` value `1279`, assigned to the international operator `"EURSLEEPER"` on 9 March 2025, started from the **VISÉ** border station with Germany, and ended at **BRUSSELS-CENTRAL**.

* Running from **Visé** German border station to **Bruxelles-Central**, it is clearly an international train coming from Germany.  

* It only operated on a specific day. There are no `"L"` train services with `"EURSLEEPER"` as operator departing from **Bruxelles-Central**, so this train appears to have disappeared. It is likely a system error which recorded this train under the `"L"` category instead of the usual `"INT"` category at the German border, then assigned it to the correct `"INT"`category when it departed from Brussels.  

### 2.2.2. Creating `Is_Local`

A new binary `Is_Local` column is created. It flags the `"S"` and `"L"` `Relation_category` classes, filtered by `"SNCB/NMBS"` in `TRAIN_SERV` to exclude the inconsistency noted above.

In [15]:
train_services["Is_Local"] = 0

In [16]:
categories_local = ["S", "L"]
mask_local = (
    (train_services["Relation_Category"].isin(categories_local)) & (train_services["TRAIN_SERV"] == "SNCB/NMBS")
    )

target_col_local = "Is_Local"

train_services.loc[mask_local, target_col_local] = 1

### 2.2.3. Creating `Is_National`

* A new binary `Is_National` column is created. It flags the `"IC"`, `"S"`, `"L"`, `"P"`, and `"EXTRA"` `Relation_Category` classes, filtered by `"SNCB/NMBS"` in `TRAIN_SERV` to exclude both `"EXTRA"` services assigned to international operators (e.g. `"EURSLEEPER"` and `"EUROSTARFR"`) and the inconsistency noted in Section 2.2.1.

* Cross-validating `TRAIN_SERV` and `Relation_Category` enables the creation of the `Is_Local` and `Is_National` columns. From these observations, it appears that `Dim_Train_Service` contains: 
    * **357** entries for local train services 
    * **127** entries for national but not local train services
    * **42** entries for international train services  

In [17]:
train_services.loc[train_services["Relation_Category"].isin(["EXTRA"])]

,TRAIN_SERV,RELATION,RELATION_DIRECTION,DIRECTION_IS_INFERRED,SK_Train_Service,Relation_Category,Is_Local
10,SNCB/NMBS,EXTRA,EXTRA,1,11,EXTRA,0
11,EUROSTARFR,EXTRA,EXTRA,1,12,EXTRA,0
12,THI-FACT,EXTRA,EXTRA,1,13,EXTRA,0


In [18]:
train_services["Is_National"] = 0

In [19]:
categories_national = ["S", "L", "IC", "P", "EXTRA"]
mask_national = (
    (train_services["Relation_Category"].isin(categories_national)) & (train_services["TRAIN_SERV"] == "SNCB/NMBS")
    )

target_col_national = "Is_National"

train_services.loc[mask_national, target_col_national] = 1

In [20]:
train_services[["Is_Local", "Is_National"]].value_counts()

Is_Local  Is_National
1         1              356
0         1              127
          0               43
Name: count, dtype: int64

## 2.3. Refining `Dim_Train_Service`

`Dim_Train_Service` is renamed and refined:  

* The columns are renamed in `Pascal Snake Case`.

* `SK_Train_service`, `Is_local`, and `Is_national` are converted to `"Int64"` to enable missing values to be mapped to `NULL` in the SQL data warehouse after loading the dimension via SQLAlchemy.

* The column order is reorganised for better clarity.

In [21]:
train_services.info()

<class 'pandas.DataFrame'>
RangeIndex: 526 entries, 0 to 525
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   TRAIN_SERV             526 non-null    string
 1   RELATION               526 non-null    string
 2   RELATION_DIRECTION     526 non-null    string
 3   DIRECTION_IS_INFERRED  526 non-null    Int64 
 4   SK_Train_Service       526 non-null    int64 
 5   Relation_Category      526 non-null    string
 6   Is_Local               526 non-null    int64 
 7   Is_National            526 non-null    int64 
dtypes: Int64(1), int64(3), string(4)
memory usage: 58.1 KB


In [22]:
Dim_Train_Service = train_services

In [23]:
Dim_Train_Service = Dim_Train_Service.rename(columns={
    "TRAIN_SERV" : "Operator_Train_Service",
    "RELATION" : "Relation",
    "RELATION_DIRECTION" : "Relation_Direction",
    "DIRECTION_IS_INFERRED" : "Direction_Is_Inferred"
})

In [24]:
Dim_Train_Service = Dim_Train_Service.astype(
    {
        "SK_Train_Service" : "Int64",
        "Is_Local" : "Int64",
        "Is_National" : "Int64"
    }
)


In [25]:
Dim_Train_Service = (Dim_Train_Service[["SK_Train_Service", 
                                  "Operator_Train_Service",
                                  "Relation_Category",
                                  "Relation",
                                  "Relation_Direction",
                                  "Direction_Is_Inferred",
                                  "Is_Local",
                                  "Is_National"]])

In [27]:
Dim_Train_Service.isnull().sum()

SK_Train_Service          0
Operator_Train_Service    0
Relation_Category         0
Relation                  0
Relation_Direction        0
Direction_Is_Inferred     0
Is_Local                  0
Is_National               0
dtype: int64

In [28]:
Dim_Train_Service.info()

<class 'pandas.DataFrame'>
RangeIndex: 526 entries, 0 to 525
Data columns (total 8 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   SK_Train_Service        526 non-null    Int64 
 1   Operator_Train_Service  526 non-null    string
 2   Relation_Category       526 non-null    string
 3   Relation                526 non-null    string
 4   Relation_Direction      526 non-null    string
 5   Direction_Is_Inferred   526 non-null    Int64 
 6   Is_Local                526 non-null    Int64 
 7   Is_National             526 non-null    Int64 
dtypes: Int64(4), string(4)
memory usage: 59.6 KB


In [29]:
Dim_Train_Service.nunique()

SK_Train_Service          526
Operator_Train_Service      4
Relation_Category          11
Relation                  171
Relation_Direction        515
Direction_Is_Inferred       2
Is_Local                    2
Is_National                 2
dtype: int64

## 2.4. Export to Gold Silver

The **train servie dimension** is now ready to be exported to the **processed directory**, as `Dim_Train_Service.parquet`, before being loaded to the SQL data warehouse.

In [30]:
Dim_Train_Service.to_parquet(PATH_PROCESSED / "Dim_Train_Service.parquet")

In [31]:
df_to_verify = pd.read_parquet(PATH_PROCESSED / "Dim_Train_Service.parquet")
df_to_verify.head()

,SK_Train_Service,Operator_Train_Service,Relation_Category,Relation,Relation_Direction,Direction_Is_Inferred,Is_Local,Is_National
0,1,SNCB/NMBS,CHARTER,CHARTER,CHARTER,1,0,0
1,2,THI-FACT,EURST,EURST,EURST: AMSTERDAM CENTRAAL -> LONDON-ST-PANCRAS...,0,0,0
2,3,THI-FACT,EURST,EURST,EURST: AMSTERDAM CENTRAAL -> MARNE-LA-VALLEE-C...,0,0,0
3,4,THI-FACT,EURST,EURST,EURST: AMSTERDAM CENTRAAL -> PARIS-NORD,0,0,0
4,5,THI-FACT,EURST,EURST,EURST: KOLN HBF -> PARIS-NORD,0,0,0


In [32]:
print(df_to_verify.shape)
df_to_verify.dtypes

(526, 8)


SK_Train_Service           Int64
Operator_Train_Service    string
Relation_Category         string
Relation                  string
Relation_Direction        string
Direction_Is_Inferred      Int64
Is_Local                   Int64
Is_National                Int64
dtype: object